In [1]:
import os
import warnings

import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.io as pio
from IPython.display import Markdown, display  # noqa: F401
from sklearn.linear_model import Lasso, LinearRegression, Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import GridSearchCV, TimeSeriesSplit

from src.preprocessing import prepare_data

warnings.filterwarnings("ignore")
pio.templates.default = "plotly_white"
pd.set_option("display.max_columns", None)
pd.set_option("display.width", None)
os.makedirs(".doc/notebook_plots/03_Regularized", exist_ok=True)

# PART 3: REGULARIZED REGRESSION - RIDGE AND LASSO

## Objective
Combat overfitting from Part 2 by applying regularization (Ridge and Lasso) with GridSearchCV optimization.

## Section 1: Load & Prepare Data

We use strategy='drop' (remove rows with missing values) to ensure a clean feature matrix, which is critical for regularization methods that penalize coefficient magnitudes — missing-value artifacts would distort the penalty.

In [2]:
data = prepare_data(strategy='drop')
df_clean = data['df_clean']
X_train = data['X_train']
X_test = data['X_test']
y_train = data['y_train']
y_test = data['y_test']
feature_names = data['feature_names']
preprocessor = data['preprocessor']

summary = pd.DataFrame({
    "Item": ["Rows after cleaning", "Train samples", "Test samples",
             "Features", "Date range start", "Date range end"],
    "Value": [
        f"{len(df_clean):,}",
        f"{X_train.shape[0]:,}",
        f"{X_test.shape[0]:,}",
        str(X_train.shape[1]),
        str(df_clean['Date'].min().date()),
        str(df_clean['Date'].max().date()),
    ],
})

display(Markdown("### Data Loading Summary"))
display(summary.style.hide(axis="index").set_caption("Dataset prepared via src.preprocessing (strategy='drop')"))

### Data Loading Summary

Item,Value
Rows after cleaning,71
Train samples,49
Test samples,22
Features,13
Date range start,2010-02-05
Date range end,2012-10-19


## Section 2: Baseline LinearRegression

An unregularized OLS baseline establishes the reference performance and overfitting level that Ridge and Lasso must improve upon.

In [3]:
baseline = LinearRegression()
baseline.fit(X_train, y_train)

y_train_pred_base = baseline.predict(X_train)
y_test_pred_base = baseline.predict(X_test)

base_train_rmse = np.sqrt(mean_squared_error(y_train, y_train_pred_base))
base_test_rmse = np.sqrt(mean_squared_error(y_test, y_test_pred_base))
base_train_r2 = r2_score(y_train, y_train_pred_base)
base_test_r2 = r2_score(y_test, y_test_pred_base)
base_overfit = (base_test_rmse - base_train_rmse) / base_train_rmse * 100

baseline_metrics = pd.DataFrame({
    "Metric": ["RMSE", "R2", "Overfitting gap"],
    "Train": [base_train_rmse, base_train_r2, None],
    "Test": [base_test_rmse, base_test_r2, None],
}).set_index("Metric")
baseline_metrics.loc["Overfitting gap", "Test"] = base_overfit

display(Markdown("### Baseline LinearRegression"))
display(baseline_metrics.style
    .format({"Train": "${:,.0f}", "Test": "${:,.0f}"}, subset=(["RMSE"], slice(None)))
    .format({"Train": "{:.4f}", "Test": "{:.4f}"}, subset=(["R2"], slice(None)))
    .format({"Test": "{:.1f}%"}, subset=(["Overfitting gap"], ["Test"]))
    .format({"Train": ""}, subset=(["Overfitting gap"], ["Train"]))
    .set_caption("Baseline Performance")
)

### Baseline LinearRegression

,Train,Test
Metric,,
RMSE,"$421,043","$1,026,115"
R2,0.6406,-1.6130
Overfitting gap,,143.7%


## Section 3: Ridge Regression with GridSearchCV

Ridge applies an L2 penalty that shrinks all coefficients toward zero without eliminating any. This is particularly effective when features are correlated (multicollinearity), as it stabilizes coefficient estimates and reduces overfitting.

In [4]:
ridge_params = {'alpha': np.logspace(-3, 3, 50)}
tscv = TimeSeriesSplit(n_splits=3)

ridge = Ridge()
ridge_grid = GridSearchCV(ridge, ridge_params, cv=tscv, scoring='neg_mean_squared_error', n_jobs=-1)
ridge_grid.fit(X_train, y_train)

ridge_best = ridge_grid.best_estimator_
y_train_pred_ridge = ridge_best.predict(X_train)
y_test_pred_ridge = ridge_best.predict(X_test)

ridge_train_rmse = np.sqrt(mean_squared_error(y_train, y_train_pred_ridge))
ridge_test_rmse = np.sqrt(mean_squared_error(y_test, y_test_pred_ridge))
ridge_train_r2 = r2_score(y_train, y_train_pred_ridge)
ridge_test_r2 = r2_score(y_test, y_test_pred_ridge)
ridge_overfit = (ridge_test_rmse - ridge_train_rmse) / ridge_train_rmse * 100

display(Markdown(
    f"### Ridge Regression — Best alpha: **{ridge_grid.best_params_['alpha']:.6f}**"
))

ridge_metrics = pd.DataFrame({
    "Metric": ["RMSE", "R2", "Overfitting gap"],
    "Train": [ridge_train_rmse, ridge_train_r2, None],
    "Test": [ridge_test_rmse, ridge_test_r2, None],
}).set_index("Metric")
ridge_metrics.loc["Overfitting gap", "Test"] = ridge_overfit

display(ridge_metrics.style
    .format({"Train": "${:,.0f}", "Test": "${:,.0f}"}, subset=(["RMSE"], slice(None)))
    .format({"Train": "{:.4f}", "Test": "{:.4f}"}, subset=(["R2"], slice(None)))
    .format({"Test": "{:.1f}%"}, subset=(["Overfitting gap"], ["Test"]))
    .format({"Train": ""}, subset=(["Overfitting gap"], ["Train"]))
    .set_caption("Ridge GridSearchCV Performance")
)

### Ridge Regression — Best alpha: **3.556480**

,Train,Test
Metric,,
RMSE,"$475,286","$820,985"
R2,0.5421,-0.6727
Overfitting gap,,72.7%


In [5]:
shrinkage_df = pd.DataFrame({
    'Feature': feature_names,
    'Baseline': baseline.coef_,
    'Ridge': ridge_best.coef_,
    'Shrinkage_%': ((1 - np.abs(ridge_best.coef_) / (np.abs(baseline.coef_) + 1e-10)) * 100).round(1)
}).sort_values('Shrinkage_%', ascending=False)

display(Markdown("### Coefficient Shrinkage: Baseline vs Ridge"))

# Median is robust to outliers where multicollinearity produces small baseline coefficients that inflate the ratio
median_shrinkage = shrinkage_df['Shrinkage_%'].median()
display(Markdown(
    f"Median shrinkage: **{median_shrinkage:.1f}%** "
    f"(median used because multicollinearity can produce small baseline coefficients whose ratios become extreme)"
))

display(shrinkage_df.style
    .format({'Baseline': '{:,.0f}', 'Ridge': '{:,.0f}', 'Shrinkage_%': '{:.1f}%'})
    .bar(subset=['Shrinkage_%'], color=['#d65f5f', '#5fba7d'], align='zero')
    .hide(axis='index')
    .set_caption("Ridge reduces coefficient magnitudes, which combats overfitting")
)

fig = go.Figure()
fig.add_trace(go.Bar(x=shrinkage_df['Feature'], y=shrinkage_df['Baseline'], name='Baseline', marker_color='steelblue'))
fig.add_trace(go.Bar(x=shrinkage_df['Feature'], y=shrinkage_df['Ridge'], name='Ridge', marker_color='coral'))
fig.update_layout(
    title='Coefficient Comparison: Baseline vs Ridge',
    yaxis_title='Coefficient Value',
    barmode='group',
    height=500,
    xaxis_tickangle=-45
)
fig.write_image(".doc/notebook_plots/03_Regularized/ridge_coefficient_shrinkage.png", scale=2)
fig.show()

### Coefficient Shrinkage: Baseline vs Ridge

Median shrinkage: **43.5%** (median used because multicollinearity can produce small baseline coefficients whose ratios become extreme)

Feature,Baseline,Ridge,Shrinkage_%
cat__Store_Grouped_19,"888,720","55,069",93.8%
cat__Store_Grouped_13,"1,313,123","388,311",70.4%
cat__Store_Grouped_999,"984,340","429,628",56.4%
num__Temperature,"51,543","-24,106",53.2%
cat__Holiday_Flag_1,"370,037","180,715",51.2%
num__Fuel_Price,"-438,180","-230,190",47.5%
num__Year,"527,412","297,788",43.5%
num__Month,"74,617","53,991",27.6%
num__Unemployment,"132,050","109,005",17.5%
cat__Store_Grouped_7,"179,097","-157,331",12.2%


### Expected output

![ridge_coefficient_shrinkage](.doc/notebook_plots/03_Regularized/ridge_coefficient_shrinkage.png)

**Observation:** Ridge shrinks most coefficients toward zero, but a few features (roughly 3 out of 13) actually increase in absolute magnitude — likely because the L2 penalty redistributes weight among correlated features (multicollinearity effect). The largest baseline coefficients (e.g. store dummies, holiday flags) retain the most magnitude. The median shrinkage percentage is more informative than the mean here because features with small baseline coefficients amplified by multicollinearity produce extreme ratios that skew the average.

## Section 4: Lasso Regression with GridSearchCV

Lasso applies an L1 penalty that can shrink coefficients exactly to zero, effectively performing feature selection. This reveals which features the model considers irrelevant and produces a sparser, more interpretable model.

In [6]:
lasso_params = {'alpha': np.logspace(-4, 3, 100)}

lasso = Lasso(max_iter=10000)
lasso_grid = GridSearchCV(lasso, lasso_params, cv=tscv, scoring='neg_mean_squared_error', n_jobs=-1)
lasso_grid.fit(X_train, y_train)

lasso_best = lasso_grid.best_estimator_
y_train_pred_lasso = lasso_best.predict(X_train)
y_test_pred_lasso = lasso_best.predict(X_test)

lasso_train_rmse = np.sqrt(mean_squared_error(y_train, y_train_pred_lasso))
lasso_test_rmse = np.sqrt(mean_squared_error(y_test, y_test_pred_lasso))
lasso_train_r2 = r2_score(y_train, y_train_pred_lasso)
lasso_test_r2 = r2_score(y_test, y_test_pred_lasso)
lasso_overfit = (lasso_test_rmse - lasso_train_rmse) / lasso_train_rmse * 100

best_alpha = lasso_grid.best_params_['alpha']
alpha_at_upper = np.isclose(best_alpha, lasso_params['alpha'].max(), rtol=0.01)
alpha_at_lower = np.isclose(best_alpha, lasso_params['alpha'].min(), rtol=0.01)

header = f"### Lasso Regression — Best alpha: **{best_alpha:.6f}**"
if alpha_at_upper or alpha_at_lower:
    bound = "upper" if alpha_at_upper else "lower"
    header += f"\n\n**WARNING:** Best alpha is at the {bound} bound of the search range — consider extending it."
display(Markdown(header))

lasso_metrics = pd.DataFrame({
    "Metric": ["RMSE", "R2", "Overfitting gap"],
    "Train": [lasso_train_rmse, lasso_train_r2, None],
    "Test": [lasso_test_rmse, lasso_test_r2, None],
}).set_index("Metric")
lasso_metrics.loc["Overfitting gap", "Test"] = lasso_overfit

display(lasso_metrics.style
    .format({"Train": "${:,.0f}", "Test": "${:,.0f}"}, subset=(["RMSE"], slice(None)))
    .format({"Train": "{:.4f}", "Test": "{:.4f}"}, subset=(["R2"], slice(None)))
    .format({"Test": "{:.1f}%"}, subset=(["Overfitting gap"], ["Test"]))
    .format({"Train": ""}, subset=(["Overfitting gap"], ["Train"]))
    .set_caption("Lasso GridSearchCV Performance")
)

### Lasso Regression — Best alpha: **1000.000000**

**WARNING:** Best alpha is at the upper bound of the search range — consider extending it.

,Train,Test
Metric,,
RMSE,"$421,452","$994,303"
R2,0.6399,-1.4535
Overfitting gap,,135.9%


## Section 5: Model Comparison

A side-by-side comparison on identical metrics (RMSE, MAE, R2, overfitting gap) quantifies whether regularization actually improved generalization over the baseline.

In [7]:
base_test_mae = mean_absolute_error(y_test, y_test_pred_base)
ridge_test_mae = mean_absolute_error(y_test, y_test_pred_ridge)
lasso_test_mae = mean_absolute_error(y_test, y_test_pred_lasso)

comparison = pd.DataFrame({
    'Model': ['Baseline', 'Ridge', 'Lasso'],
    'Train RMSE': [base_train_rmse, ridge_train_rmse, lasso_train_rmse],
    'Test RMSE': [base_test_rmse, ridge_test_rmse, lasso_test_rmse],
    'Test MAE': [base_test_mae, ridge_test_mae, lasso_test_mae],
    'Test R2': [base_test_r2, ridge_test_r2, lasso_test_r2],
    'Overfitting %': [
        (base_test_rmse - base_train_rmse) / base_train_rmse * 100,
        (ridge_test_rmse - ridge_train_rmse) / ridge_train_rmse * 100,
        (lasso_test_rmse - lasso_train_rmse) / lasso_train_rmse * 100,
    ]
}).set_index('Model')

display(Markdown("### Model Comparison"))
display(comparison.style
    .format({
        'Train RMSE': '${:,.0f}',
        'Test RMSE': '${:,.0f}',
        'Test MAE': '${:,.0f}',
        'Test R2': '{:.4f}',
        'Overfitting %': '{:.1f}%',
    })
    .highlight_min(subset=['Test RMSE', 'Test MAE', 'Overfitting %'], color='#c6efce')
    .highlight_max(subset=['Test R2'], color='#c6efce')
    .set_caption("All models evaluated on identical TimeSeriesSplit CV")
)

### Model Comparison

,Train RMSE,Test RMSE,Test MAE,Test R2,Overfitting %
Model,,,,,
Baseline,"$421,043","$1,026,115","$923,021",-1.6130,143.7%
Ridge,"$475,286","$820,985","$670,398",-0.6727,72.7%
Lasso,"$421,452","$994,303","$889,794",-1.4535,135.9%


## Section 6: Lasso Feature Selection

L1 regularization drives irrelevant coefficients exactly to zero. Examining which features survive reveals the most predictive signals and quantifies dimensionality reduction.

In [8]:
feature_names_arr = np.array(feature_names)

lasso_zero = (lasso_best.coef_ == 0).sum()
lasso_nonzero = (lasso_best.coef_ != 0).sum()

display(Markdown("### Lasso Feature Selection"))

selection_summary = pd.DataFrame({
    "Item": ["Total features", "Eliminated (coef=0)", "Selected (coef!=0)", "Feature reduction"],
    "Value": [
        len(lasso_best.coef_),
        lasso_zero,
        lasso_nonzero,
        f"{lasso_zero / len(lasso_best.coef_) * 100:.1f}%",
    ],
})
display(selection_summary.style.hide(axis="index"))

if lasso_nonzero < len(lasso_best.coef_):
    selected = pd.DataFrame({
        'Feature': feature_names_arr[lasso_best.coef_ != 0],
        'Coefficient': lasso_best.coef_[lasso_best.coef_ != 0]
    }).sort_values('Coefficient', key=abs, ascending=False)

    display(Markdown("**Selected features:**"))
    display(selected.style
        .format({'Coefficient': '{:,.0f}'})
        .hide(axis='index')
    )
else:
    display(Markdown(
        "No features eliminated by Lasso at this alpha. "
        "The best alpha hit the upper bound — stronger regularization may eliminate features."
    ))

### Lasso Feature Selection

Item,Value
Total features,13
Eliminated (coef=0),0
Selected (coef!=0),13
Feature reduction,0.0%


No features eliminated by Lasso at this alpha. The best alpha hit the upper bound — stronger regularization may eliminate features.

## Section 7: Visualizations

Visual comparison of RMSE, R2, and overfitting gaps makes it easier to spot whether regularization improved generalization or merely shifted the train-test trade-off.

In [9]:
fig1 = go.Figure()

models = ['Baseline', 'Ridge', 'Lasso']
train_rmses = [base_train_rmse, ridge_train_rmse, lasso_train_rmse]
test_rmses = [base_test_rmse, ridge_test_rmse, lasso_test_rmse]

fig1.add_trace(go.Bar(x=models, y=train_rmses, name='Train RMSE', marker_color='steelblue'))
fig1.add_trace(go.Bar(x=models, y=test_rmses, name='Test RMSE', marker_color='coral'))

fig1.update_layout(
    title='RMSE Comparison: Baseline vs Regularized Models',
    yaxis_title='RMSE ($)',
    barmode='group',
    height=500
)
fig1.write_image(".doc/notebook_plots/03_Regularized/rmse_comparison.png", scale=2)
fig1.show()

### Expected output

![rmse_comparison](.doc/notebook_plots/03_Regularized/rmse_comparison.png)

In [10]:
fig2 = go.Figure()

train_r2s = [base_train_r2, ridge_train_r2, lasso_train_r2]
test_r2s = [base_test_r2, ridge_test_r2, lasso_test_r2]

fig2.add_trace(go.Bar(x=models, y=train_r2s, name='Train R2', marker_color='lightblue'))
fig2.add_trace(go.Bar(x=models, y=test_r2s, name='Test R2', marker_color='lightcoral'))

fig2.update_layout(
    title='R2 Score Comparison',
    yaxis_title='R2 Score',
    barmode='group',
    height=500,
    yaxis=dict(range=[-2.5, 1])
)
fig2.write_image(".doc/notebook_plots/03_Regularized/r2_score_comparison.png", scale=2)
fig2.show()

### Expected output

![r2_score_comparison](.doc/notebook_plots/03_Regularized/r2_score_comparison.png)

In [11]:
gaps = [
    (base_test_rmse - base_train_rmse)/base_train_rmse*100,
    (ridge_test_rmse - ridge_train_rmse)/ridge_train_rmse*100,
    (lasso_test_rmse - lasso_train_rmse)/lasso_train_rmse*100
]

colors = ['red' if gap > 15 else 'orange' if gap > 5 else 'green' for gap in gaps]

fig3 = go.Figure(data=go.Bar(x=models, y=gaps, marker_color=colors))
fig3.add_hline(y=5, line_dash="dash", line_color="green")
fig3.add_hline(y=15, line_dash="dash", line_color="orange")

fig3.update_layout(
    title='Overfitting Gap Comparison',
    yaxis_title='RMSE Gap (%)',
    height=500
)
fig3.write_image(".doc/notebook_plots/03_Regularized/overfitting_gap.png", scale=2)
fig3.show()

### Expected output

![overfitting_gap](.doc/notebook_plots/03_Regularized/overfitting_gap.png)

**Observation (charts above):** RMSE charts show Ridge achieves a meaningful ~20% improvement over the baseline, while Lasso performs comparably to OLS. Despite this, all models retain negative R2, confirming that regularization alone cannot rescue a fundamentally linear model on this non-linear dataset. The overfitting gap chart highlights that Ridge roughly halves the train-test gap compared to the baseline, but all models remain above the 15% threshold (red zone), indicating persistent generalization failure.

## Section 8: Final Summary

Consolidates all results into a single table to support the go/no-go decision on linear regularization and guide the choice of next modeling approach.

In [12]:
improvement_ridge = (base_test_rmse - ridge_test_rmse) / base_test_rmse * 100
improvement_lasso = (base_test_rmse - lasso_test_rmse) / base_test_rmse * 100
best_model = comparison['Test RMSE'].idxmin()
best_rmse = comparison.loc[best_model, 'Test RMSE']
best_r2 = comparison.loc[best_model, 'Test R2']
best_overfit = comparison.loc[best_model, 'Overfitting %']

display(Markdown(f"""### Part 3 -- Final Summary

**Dataset:** {len(df_clean):,} samples ({X_train.shape[0]:,} train / {X_test.shape[0]:,} test), \
{X_train.shape[1]} features, sample-to-feature ratio {X_train.shape[0] / X_train.shape[1]:.1f}:1

| Model | Test RMSE | Test MAE | Test R2 | Overfitting | vs Baseline |
|-------|-----------|----------|---------|-------------|-------------|
| Baseline | ${base_test_rmse:,.0f} | ${base_test_mae:,.0f} | {base_test_r2:.4f} | {base_overfit:.1f}% | -- |
| Ridge (alpha={ridge_grid.best_params_['alpha']:.4f}) | ${ridge_test_rmse:,.0f} | ${ridge_test_mae:,.0f} | {ridge_test_r2:.4f} | {ridge_overfit:.1f}% | {improvement_ridge:+.1f}% |
| Lasso (alpha={lasso_grid.best_params_['alpha']:.4f}) | ${lasso_test_rmse:,.0f} | ${lasso_test_mae:,.0f} | {lasso_test_r2:.4f} | {lasso_overfit:.1f}% | {improvement_lasso:+.1f}% |

**Best model: {best_model}** with Test RMSE ${best_rmse:,.0f}, R2 {best_r2:.4f}, \
overfitting gap {best_overfit:.1f}%.

**Regularization impact:** Ridge reduced overfitting from {base_overfit:.1f}% to {ridge_overfit:.1f}% \
and improved RMSE by {improvement_ridge:.1f}%. \
Lasso eliminated {lasso_zero}/{len(lasso_best.coef_)} features \
and improved RMSE by {improvement_lasso:.1f}%.

**Conclusion:** All models show negative R2 and high overfitting, indicating the linear approach \
is insufficient for this dataset. Non-linear models (tree-based, etc.) should be explored next.
"""))

### Part 3 -- Final Summary

**Dataset:** 71 samples (49 train / 22 test), 13 features, sample-to-feature ratio 3.8:1

| Model | Test RMSE | Test MAE | Test R2 | Overfitting | vs Baseline |
|-------|-----------|----------|---------|-------------|-------------|
| Baseline | $1,026,115 | $923,021 | -1.6130 | 143.7% | -- |
| Ridge (alpha=3.5565) | $820,985 | $670,398 | -0.6727 | 72.7% | +20.0% |
| Lasso (alpha=1000.0000) | $994,303 | $889,794 | -1.4535 | 135.9% | +3.1% |

**Best model: Ridge** with Test RMSE $820,985, R2 -0.6727, overfitting gap 72.7%.

**Regularization impact:** Ridge reduced overfitting from 143.7% to 72.7% and improved RMSE by 20.0%. Lasso eliminated 0/13 features and improved RMSE by 3.1%.

**Conclusion:** All models show negative R2 and high overfitting, indicating the linear approach is insufficient for this dataset. Non-linear models (tree-based, etc.) should be explored next.
